# 10 · Nonlinear problems — Allen–Cahn & Newton

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "anywidget"], check=True)

In [ ]:
from ngsolve import *
from ngsolve import solvers
from netgen.occ import WorkPlane, OCCGeometry, X, Y, IdentificationType
from ngsolve.webgui import Draw
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def progress(i, n):                                    # a tiny dependency-free bar (live frontends only)
    import os, sys
    if os.environ.get("WEBGUI_SCENE_DIR"):             # static-site build: stay silent (no \r in the HTML)
        return
    if (i + 1) % max(1, n // 100) == 0 or i + 1 == n:
        b = int(28 * (i + 1) / n)
        sys.stdout.write(f"\r  marching… [{'█'*b}{'·'*(28-b)}] {100*(i+1)//n:3d}%"); sys.stdout.flush()
        if i + 1 == n:
            sys.stdout.write("\n")

## 1. The Allen–Cahn model

A field $u(t,\mathbf x)\in[-1,1]$ labels two **phases** ($u=\pm1$). It flows downhill in the
**Ginzburg–Landau energy** 
$$E(u)=\int \tfrac\gamma2|\nabla u|^2+\tfrac1\gamma W(u),$$ with the
**double-well** $W(u)=\tfrac14(u^2-1)^2$ (two minima at $\pm1$). The gradient flow is
$$ \partial_t u \;=\; \gamma\,\Delta u \;+\; \tfrac1\gamma\,(u-u^3) , $$

* We run it on a **periodic** square, so **no boundary conditions** are needed. 
* The reaction
$u-u^3=-W'(u)$ pushes $u$ toward $\pm1$; the $\gamma\Delta u$
smooths interfaces of width $\sim\gamma$.
* The **$u^3$ is the nonlinearity** — that is what makes
the implicit step a *nonlinear* solve.

In [ ]:
# a PERIODIC unit square: identify left<->right and bottom<->top, so the domain wraps around
square = WorkPlane().Rectangle(1, 1).Face()
square.edges.Min(X).name = "left";   square.edges.Max(X).name = "right"
square.edges.Min(Y).name = "bottom"; square.edges.Max(Y).name = "top"
square.edges.Min(X).Identify(square.edges.Max(X), "leftright", IdentificationType.PERIODIC)
square.edges.Min(Y).Identify(square.edges.Max(Y), "bottomtop", IdentificationType.PERIODIC)
mesh = Mesh(OCCGeometry(square, dim=2).GenerateMesh(maxh=0.1))

Next, we want to set initial values through a structured table of data:

We take the `VoxelCoefficient` to put structured data into a `CF`: 

In [ ]:
# random initial phase, set from a VoxelCoefficient on a grid FINER than the FE resolution
# h/k = 0.1/3 ≈ 0.033, so 40×40 voxels (spacing 0.025) sit below it; linear=False -> cell-wise random
np.random.seed(3)
randvals = 0.1 * (np.random.rand(3, 3) - 0.5) # ranges from -0.05 to 0.05
randCF = VoxelCoefficient((0, 0), (1, 1), randvals, linear=False)
view = dict(euler_angles=[-45.68822825404629,1.4679271675638002,1.011669327261409])
Draw(randCF, mesh, "rand", min=-0.05, max=0.05, autoscale=False, deformation=True, **view)

Note: continuity is only a visualization artefact!

In [ ]:
# the periodic FESpace using the Periodic(…) wrapper:
fes = Periodic(H1(mesh, order=3))                      # periodic -> no boundary dofs at all
u, v = fes.TnT()
gfu = GridFunction(fes); uold = GridFunction(fes)

# random initial phase, set from a VoxelCoefficient on a grid FINER than the FE resolution
# h/k = 0.1/3 ≈ 0.033, so 40×40 voxels (spacing 0.025) sit below it; linear=False -> cell-wise random
np.random.seed(3)
randvals = 0.1 * (np.random.rand(40, 40) - 0.5) # ranges from -0.05 to 0.05
gfu.Set(VoxelCoefficient((0, 0), (1, 1), randvals, linear=True))

Draw(gfu, mesh, "phase u", min=-0.05, max=0.05, autoscale=False, deformation=True, **view)

## 2. One implicit step is nonlinear — solve it with Newton

Implicit Euler for the step $u^n\to u^{n+1}\equiv u$ gives the **residual**
$$ F(u)\;=\;\int \frac{u-u^n}{\Delta t}\,v \;+\; \gamma\,\nabla u\!\cdot\!\nabla v
   \;-\; \tfrac1\gamma(u-u^3)\,v \;\;\mathrm dx \;\overset!=\;0 \quad\forall v, $$
**nonlinear in $u$** through the $u^3$. 
We write $F$ as a `BilinearForm` (the trial function `u` *is* the unknown), and **`solvers.Newton`** does the rest: 
* at each iterate it assembles the **Jacobian** $F'(u)$ — NGSolve differentiates the form **automatically** (see `Diff`) —
* solves $F'(u)\,\delta = -F(u)$,
* updates $u\mathrel{+}=\delta$,
* and repeats until (numerical) convergence

In [ ]:
gamma, dt = 0.03, 0.004                                # interface width ~gamma, time step

In [ ]:
a = BilinearForm(fes)
a += (1/dt)*(u - uold)*v*dx + gamma*grad(u)*grad(v)*dx - (1/gamma)*(u - u**3)*v*dx

uold.vec.data = gfu.vec
solvers.Newton(a, gfu, printing=True, maxit=20)        # ONE nonlinear step, Newton iterations shown

We *derived* $F$ from the energy by hand. We need not: 
* hand NGSolve the **energy** of the implicit step (via **`Variation`**)
    * and it forms the residual (1st variation)
    * *and* Jacobian (2nd variation) **automatically**

$\leadsto$ same Newton, same step, no hand-derivation.

In [ ]:
aE = BilinearForm(fes, symmetric=True)
aE += Variation((0.5/dt*(u - uold)**2                                              # implicit-Euler step
                 + 0.5*gamma*grad(u)*grad(u) + (1/gamma)*0.25*(u**2 - 1)**2) * dx)  # minimises this energy
gfE = GridFunction(fes); gfE.vec.data = uold.vec
solvers.Newton(aE, gfE, printing=True, maxit=20)
print("energy form vs hand-rolled residual:  max |Δ| = "
      f"{max(abs(gfu.vec[i] - gfE.vec[i]) for i in range(len(gfu.vec))):.1e}")

## 3. March in time — watch the phases coarsen

Repeat the nonlinear step. From the random seed the field **separates** into $\pm1$ patches
(spinodal decomposition), then the patches **coarsen** — interfaces shrink to lower the energy.

In [ ]:
nsteps = 150
anim = GridFunction(fes, multidim=0)
anim.AddMultiDimComponent(gfu.vec)
energy = []
with TaskManager():
    for step in range(1, nsteps + 1):
        uold.vec.data = gfu.vec
        solvers.Newton(a, gfu, printing=False, maxit=20)
        if step % (nsteps//12) == 0:                             # ~12 snapshots for the animation + energy
            anim.AddMultiDimComponent(gfu.vec)
            energy.append(Integrate(0.5*gamma*grad(gfu)*grad(gfu) + (1/gamma)*0.25*(gfu*gfu-1)**2, mesh))
        progress(step - 1, nsteps)

In [ ]:
Draw(anim, mesh, "phase u", min=-1, max=1, autoscale=False,
     interpolate_multidim=True, animate=True, order=3, deformation=True, **view)

The energy decay:

In [ ]:
plt.figure(figsize=(7, 2.6))
plt.plot([12*(i+1)*dt for i in range(len(energy))], energy, "-o", ms=3)
plt.xlabel("time"); plt.ylabel("Ginzburg–Landau energy"); plt.grid(alpha=0.3)
plt.title("energy decreases — a gradient flow"); plt.tight_layout()

**Next:** combine **unsteady** (unit 8) and **nonlinear** (this unit) and add a *second* reacting
species — and patterns grow themselves on the Beast's skin (Part III, Turing).

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("09-dg-hdg", "9 · Discontinuous Galerkin & HDG")
    _next = ("11-turing-patterns", "11 · A different pattern: encounter with a relative 🧬")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))